# Signed Jumps 05 — does the jump hazard add to VD 19's model-versus-implied signal?

Protocol: `discussions/Signed_Jumps/011_claude_sj05_preregistration.md` (fixed before this run).

Reads VD 19 v0.1.1 rows (unchanged) → SJ 01 expected jumps from entry to the 08:00 expiry (model (e), no refit) →
$y_{rv,index}=a+\beta x+\gamma x_J$ at the 04:00 entry with VD 19's session bootstrap. Gate: γ lower bound > 0.

In [1]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import signed_jumps as sj, signed_jumps_events as se, signed_jumps_vd19 as sv, model_vs_implied as mv
for m in (sj,se,sv):importlib.reload(m)
assert sv.VERSION=='SJ05-0.1.0' and mv.VERSION.startswith('VD19-0.1.1')
SYNTHETIC=os.getenv('SJ05_SYNTHETIC','0')=='1'
p=sj.Params();vp=sv.VdParams()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='sj05-');OUT=Path(tmp.name)/'06'
else:
    OUT=ROOT/'notebooks'/'Signed_Jumps'/'06'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
if not SYNTHETIC:
    chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_signed_jumps_vd19.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
    print(chk.stdout[-600:]);print(chk.stderr[-400:]);assert chk.returncode==0,'tests failed'
print(json.dumps(sv.registration(vp),indent=1))

......                                                                   [100%]
6 passed in 11.09s


{
 "primary_entry_hour": 4,
 "primary_outcome": "y_rv_index",
 "secondary_outcomes": [
  "y_rv",
  "y"
 ],
 "vd19_selection_log_ratio": 0.22314355131420976,
 "origin_tolerance_min": 5,
 "status": "PREREGISTERED_SJ011",
 "version": "SJ05-0.1.0",
 "gate": "entry 04:00, y_rv_index ~ x + xJ, VD19 session bootstrap: gamma lo > 0 => JUMP_HAZARD_ADDS_INFORMATION; gamma hi < 0 => JUMP_HAZARD_ANTI; else NO_EVIDENCE"
}


## VD 19 rows (read only) and the SJ 01 jump hazard to expiry

In [2]:
T0=time.time()
if SYNTHETIC:
    D=sv.synthetic_rows(n_sessions=25,gamma=0.5,seed=2)
else:
    rows=pd.read_csv(ROOT/'notebooks'/'Volatility_Dynamics'/'19'/'19_rows.csv')
    print('VD19 rows:',rows.shape,'| sessions',rows.session.min(),'->',rows.session.max());display(rows.groupby('entry_hour').size())
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
    S={A:sj.load_spot_1m(BIN/'basis'/'spot'/f'{A}USDT_spot_1m.parquet') for A in p.assets}
    assets,pan,_=sj.prepare(S,p)
    fits=se.fits_from_table(pd.read_csv(ROOT/'notebooks'/'Signed_Jumps'/'01'/'01_fits.csv'),pan,p,'e')
    orig=sv.origin_rows(pan,rows,vp)
    print('origins found:',int((orig>=0).sum()),'of',len(rows))
    lam,lam_b=sv.hazard_to_expiry(fits,assets,pan,p,orig,rows['h'].to_numpy(),'e')
    D=sv.attach(rows,lam,lam_b)
    D[['session','underlying','entry_hour','h','x','Lambda','Lambda_base','xJ','y_rv_index','y_rv','y','R_net']].to_csv(OUT/'06_rows.csv',index=False)
    display(D.groupby('entry_hour')[['Lambda','Lambda_base','xJ','x']].describe().T.round(3))
print(f'{time.time()-T0:.0f}s')

VD19 rows: (244, 14) | sessions 2026-09-21 -> 2026-10-13


entry_hour
4     85
6     73
12    86
dtype: int64

D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:172: RuntimeWarning: divide by zero encountered in log
  feats.append(np.log(m))
D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:187: RuntimeWarning: divide by zero encountered in divide
  Z = z / np.sqrt(sig2c)
D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:187: RuntimeWarning: invalid value encountered in divide
  Z = z / np.sqrt(sig2c)


origins found: 244 of 244


entry_hour             4       6       12
Lambda      count  85.000  73.000  86.000
            mean    0.679   0.316   3.300
            std     0.915   0.228   2.227
            min     0.132   0.043   0.556
            25%     0.324   0.166   1.974
            50%     0.456   0.236   2.726
            75%     0.694   0.440   4.146
            max     6.198   1.018  11.401
Lambda_base count  85.000  73.000  86.000
            mean    0.677   0.316   3.300
            std     0.915   0.228   2.227
            min     0.132   0.043   0.556
            25%     0.324   0.166   1.974
            50%     0.456   0.236   2.726
            75%     0.680   0.440   4.146
            max     6.198   1.018  11.401
xJ          count  85.000  73.000  86.000
            mean   -0.708  -1.372   1.008
            std     0.684   0.661   0.610
            min    -2.022  -3.148  -0.587
            25%    -1.128  -1.796   0.680
            50%    -0.786  -1.446   1.003
            75%    -0.366  -0.821   1.422
            max     1.824   0.018   2.434
x           count  85.000  73.000  86.000
            mean    0.039   0.046   0.145
            std     0.202   0.236   0.208
            min    -0.452  -0.667  -0.450
            25%    -0.119  -0.111  -0.007
            50%     0.042   0.056   0.131
            75%     0.188   0.205   0.295
            max     0.486   0.696   0.637

161s


## Information: $y=a+\beta x+\gamma x_J$ (and $x_J$ alone)

In [3]:
info=sv.information(D,mv,vp);info.to_csv(OUT/'06_information.csv',index=False)
cols=[c for c in ['entry_hour','outcome','model','n','sessions','corr_x_xJ','x','x_lo','x_hi','xJ','xJ_lo','xJ_hi'] if c in info]
display(info[cols].round(3))

,entry_hour,outcome,model,n,sessions,corr_x_xJ,x,x_lo,x_hi,xJ,xJ_lo,xJ_hi
0,4,y_rv_index,x+xJ,77,21,-0.163,1.500,0.490,2.453,0.101,-0.134,0.627
1,4,y_rv_index,xJ_only,77,21,-0.163,NaN,NaN,NaN,0.045,-0.253,0.652
2,4,y_rv_index,x_only,77,21,-0.163,1.453,0.551,2.402,NaN,NaN,NaN
3,4,y_rv,x+xJ,85,23,-0.163,1.290,0.432,2.219,0.105,-0.131,0.584
4,4,y_rv,xJ_only,85,23,-0.163,NaN,NaN,NaN,0.043,-0.232,0.604
5,4,y_rv,x_only,85,23,-0.163,1.233,0.368,2.172,NaN,NaN,NaN
6,4,y,x+xJ,85,23,-0.163,0.555,-1.026,1.828,0.202,-0.117,0.815
7,4,y,xJ_only,85,23,-0.163,NaN,NaN,NaN,0.175,-0.174,0.818
8,4,y,x_only,85,23,-0.163,0.443,-0.987,1.756,NaN,NaN,NaN
9,6,y_rv_index,x+xJ,66,21,0.007,1.255,0.509,1.984,-0.077,-0.384,0.282


## Economics (descriptive): VD 19-selected straddles split by jump hazard

In [4]:
if 'R_net' in D and D['R_net'].notna().any():
    eco=sv.economics(D,mv,vp);eco.to_csv(OUT/'06_economics.csv',index=False);display(eco.round(3))

,entry_hour,set,half,xJ_median,n_sessions,n,mean,lo,hi
0,4,vd19_selected,xJ_high,-0.786,7,9,-0.022,-0.544,0.574
1,4,vd19_selected,xJ_low,-0.786,6,10,-0.407,-0.770,-0.026
2,4,all,xJ_high,-0.786,20,43,-0.377,-0.607,-0.098
3,4,all,xJ_low,-0.786,18,42,-0.402,-0.584,-0.205
4,6,vd19_selected,xJ_high,-1.446,6,7,-0.062,-0.598,0.523
5,6,vd19_selected,xJ_low,-1.446,6,8,0.069,-0.332,0.490
6,6,all,xJ_high,-1.446,18,37,-0.413,-0.638,-0.127
7,6,all,xJ_low,-1.446,15,36,-0.331,-0.551,-0.093
8,12,vd19_selected,xJ_high,1.003,11,16,-0.326,-0.568,-0.086
9,12,vd19_selected,xJ_low,1.003,11,16,-0.201,-0.514,0.140


## Decision

In [5]:
dec=sv.decide(info,vp);dec['synthetic']=SYNTHETIC
print(json.dumps(dec,indent=1,default=float))
if not SYNTHETIC:
    (OUT/'06_decision.json').write_text(json.dumps(dec,indent=1,default=float))
    (OUT/'06_manifest.json').write_text(json.dumps({'registration':sv.registration(vp),'runtime_s':time.time()-T0},indent=1,default=str))

{
 "version": "SJ05-0.1.0",
 "verdict": "NO_EVIDENCE",
 "gamma": 0.1011171316561721,
 "gamma_ci": [
  -0.1342329350036448,
  0.6266642714879387
 ],
 "beta": 1.4999230525753848,
 "beta_ci": [
  0.49007470965983846,
  2.452569894288886
 ],
 "n": 77,
 "sessions": 21,
 "corr_x_xJ": -0.1628078054393665,
 "status": "PREREGISTERED_SJ011",
 "prospective_confirmation": false,
 "synthetic": false
}
